In [1]:
import utils
import numpy as np
import pandas as pd
import os
import re

import matplotlib.pyplot as plt
import seaborn as sns # Loads in the theme
sns.set_theme()

pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_rows', None)

In [2]:
# Make output folders
output_dir = os.path.join('/Volumes/project/3025011.02/pre-processed/behaviour')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [3]:
# Preparation
pilot_analysis = False
unique_subject_ids = [5]#[802,803,804,805,806,807,808,809,810,811,813,814,815]
unique_sessions = [2, 3, 4]
plot_range = [0, 6]

if pilot_analysis:
    raw_output_path = '/Volumes/project/3025011.02/pilot_data/phd_1_task/experiment_output/behavioural_files_pilot_2'
else:
    raw_output_path = '/Volumes/project/3025011.02/raw/'
recent_results = utils.analysis_functions.list_files_with_date_and_subject_id(raw_output_path, unique_subject_ids, unique_sessions)

In [4]:
if not recent_results:
    raise Exception('No files found')
print(recent_results)

['/Volumes/project/3025011.02/raw/sub-005/ses-mri02/beh/behavioural_output_sub-005_session-02_skyra_2025-03-14_11-00-23.csv', '/Volumes/project/3025011.02/raw/sub-005/ses-mri03/beh/behavioural_output_sub-005_session-03_skyra_2025-03-21_15-48-02.csv', '/Volumes/project/3025011.02/raw/sub-005/ses-mri04/beh/behavioural_output_sub-005_session-04_skyra_2025-03-28_15-50-17.csv']


In [5]:
### Remove invalid files
# Remove files that contain invalid data
raw_output_path = '/Volumes/project/3025011.02/raw/'
invalid_data = pd.read_csv(f'{raw_output_path}../incomplete_data.csv', delimiter=';')
combinations = {
    f"sub-{row['subject-id']:03}_session-{row['session-number']:02}"
    for _, row in invalid_data[invalid_data['datatype'] == 'beh'].iterrows()
}
print(f'These participants will be excluded: {combinations}')

patterns = [re.compile(re.escape(combo)) for combo in combinations]
# Filter the list: remove any string that contains one of the combinations
recent_results = [
    item for item in recent_results
    if not any(pattern.search(item) for pattern in patterns)
]
print(recent_results)

These participants will be excluded: {'sub-003_session-03'}
['/Volumes/project/3025011.02/raw/sub-005/ses-mri02/beh/behavioural_output_sub-005_session-02_skyra_2025-03-14_11-00-23.csv', '/Volumes/project/3025011.02/raw/sub-005/ses-mri03/beh/behavioural_output_sub-005_session-03_skyra_2025-03-21_15-48-02.csv', '/Volumes/project/3025011.02/raw/sub-005/ses-mri04/beh/behavioural_output_sub-005_session-04_skyra_2025-03-28_15-50-17.csv']


In [6]:
combined_results_dataframe = utils.analysis_functions.combine_result_files(recent_results)
if pilot_analysis:
    combined_results_dataframe.rename(columns={'iti': 'trial_duration'}, inplace=True)
    combined_results_dataframe['stimuli_type'] = combined_results_dataframe['stimuli_type'].replace(3, 2)
print(combined_results_dataframe)

      trial  emotional_cue_time response objectively_correct  \
0         0        1.741947e+09     down                Even   
1         1        1.741947e+09       up                Even   
2         2        1.741947e+09       up                Even   
3         3        1.741947e+09     down                Even   
4         4        1.741947e+09     down                Even   
5         5        1.741947e+09     down                Even   
6         6        1.741947e+09       up                Even   
7         7        1.741947e+09     down                Even   
8         8        1.741947e+09       up               False   
9         9        1.741947e+09       up                True   
10       10        1.741947e+09     down               False   
11       11        1.741947e+09       up               False   
12       12        1.741947e+09       up               False   
13       13        1.741947e+09       up                True   
14       14        1.741947e+09     down

# Flip results if joystick orientation was incorrect
Specifically, switch the response and probability

In [7]:
# List of subject_session combinations to update
match_list = ['sub-004_session-04']

# Create a new column that combines subject_id and session in the same format as the list.
combined_results_dataframe['sub_session'] = combined_results_dataframe['subject_id'] + '_' + combined_results_dataframe['session']

# Create a mask for rows that match any of the combinations in match_list.
mask = combined_results_dataframe['sub_session'].isin(match_list)

# For 'probability_condition', swap 80 with 20 and vice versa.
combined_results_dataframe.loc[mask, 'probability_condition'] = (
    combined_results_dataframe.loc[mask, 'probability_condition']
    .replace({80: 20, 20: 80})
)

# For 'response', swap 'up' with 'down' and vice versa.
combined_results_dataframe.loc[mask, 'response'] = (
    combined_results_dataframe.loc[mask, 'response']
    .replace({'up': 'down', 'down': 'up'})
)

# Optionally, drop the helper column
combined_results_dataframe.drop(columns='sub_session', inplace=True)

print(combined_results_dataframe)

      trial  emotional_cue_time response objectively_correct  \
0         0        1.741947e+09     down                Even   
1         1        1.741947e+09       up                Even   
2         2        1.741947e+09       up                Even   
3         3        1.741947e+09     down                Even   
4         4        1.741947e+09     down                Even   
5         5        1.741947e+09     down                Even   
6         6        1.741947e+09       up                Even   
7         7        1.741947e+09     down                Even   
8         8        1.741947e+09       up               False   
9         9        1.741947e+09       up                True   
10       10        1.741947e+09     down               False   
11       11        1.741947e+09       up               False   
12       12        1.741947e+09       up               False   
13       13        1.741947e+09       up                True   
14       14        1.741947e+09     down

# Remove non-responses

In [8]:
mask = ~(combined_results_dataframe['response'].isna() | combined_results_dataframe['objectively_correct'].str.contains('Late', na=False))
combined_results_dataframe = combined_results_dataframe[mask]

# Check congruency, valence and volatility

In [9]:
# Check a block's congruency
combined_results_dataframe['congruency'] = combined_results_dataframe.apply(utils.analysis_functions.check_congruency, args=(True,), axis=1)

# Check a block's valence
combined_results_dataframe['valence'] = combined_results_dataframe.apply(utils.analysis_functions.check_valence, args=(True,), axis=1)

# Add WSLS columns
combined_results_dataframe = utils.learning_models.add_WSLS_column(combined_results_dataframe)

# Check a block's volatility
combined_results_dataframe = combined_results_dataframe[combined_results_dataframe['probability_condition'] != 50]
combined_results_dataframe = combined_results_dataframe.groupby('stimuli_type', group_keys=False).apply(lambda x: utils.analysis_functions.check_volatility(x, True))
combined_results_dataframe = combined_results_dataframe.sort_values(by=['stimuli_type','trial'], ascending=[True, True])
print(combined_results_dataframe[['volatility','probability_condition']])
combined_results_dataframe = combined_results_dataframe.sort_values(by=['subject_id', 'session', 'stimuli_type', 'trial'], ascending=[True, True, True, True])
combined_results_dataframe.reset_index(drop=True, inplace=True)
print(combined_results_dataframe[['subject_id', 'session', 'congruency', 'valence', 'volatility']])
print(combined_results_dataframe.shape[0])

print(combined_results_dataframe['valence'].unique())
print(combined_results_dataframe['congruency'].unique())
print(combined_results_dataframe['volatility'].unique())

     subjectively_correct_one_back response_one_back response  WS  LS
0                              NaN               NaN     down   0   0
1                            False              down       up   0   1
2                             True                up       up   1   0
3                             True                up     down  -1   0
4                            False              down       up   0   1
5                            False                up       up   0  -1
6                            False                up       up   0  -1
7                            False                up     down   0   1
8                             True              down     down   1   0
9                             True              down     down   1   0
10                            True              down       up  -1   0
11                           False                up     down   0   1
12                           False              down     down   0  -1
13                  

/var/folders/bh/ld02p1_550d75y0x32dmpb540000gp/T/ipykernel_7033/19396455.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_results_dataframe = combined_results_dataframe.groupby('stimuli_type', group_keys=False).apply(lambda x: utils.analysis_functions.check_volatility(x, True))


In [10]:
print(combined_results_dataframe)

      trial  emotional_cue_time response objectively_correct  \
0         8        1.741947e+09       up               False   
1        11        1.741947e+09       up               False   
2        12        1.741947e+09       up               False   
3        14        1.741947e+09     down                True   
4        15        1.741947e+09     down                True   
5        18        1.741947e+09     down                True   
6        19        1.741947e+09       up               False   
7        23        1.741947e+09     down                True   
8        24        1.741947e+09     down                True   
9        26        1.741947e+09     down                True   
10       27        1.741947e+09     down                True   
11       32        1.741947e+09     down                True   
12       33        1.741947e+09     down                True   
13       35        1.741947e+09     down                True   
14       36        1.741947e+09     down

# Now add dummy variables for the stimulation condition

In [11]:
combined_results_dataframe['session_2'] = (combined_results_dataframe['session'] == 'session-02').astype(int)
combined_results_dataframe['session_3'] = (combined_results_dataframe['session'] == 'session-03').astype(int)
combined_results_dataframe['session_4'] = (combined_results_dataframe['session'] == 'session-04').astype(int)
print(combined_results_dataframe[['session','session_2', 'session_3']])

         session  session_2  session_3
0     session-02          1          0
1     session-02          1          0
2     session-02          1          0
3     session-02          1          0
4     session-02          1          0
5     session-02          1          0
6     session-02          1          0
7     session-02          1          0
8     session-02          1          0
9     session-02          1          0
10    session-02          1          0
11    session-02          1          0
12    session-02          1          0
13    session-02          1          0
14    session-02          1          0
15    session-02          1          0
16    session-02          1          0
17    session-02          1          0
18    session-02          1          0
19    session-02          1          0
20    session-02          1          0
21    session-02          1          0
22    session-02          1          0
23    session-02          1          0
24    session-02         

# Recode existing variables

In [12]:
combined_results_dataframe = combined_results_dataframe.dropna(subset=['response'])

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'up': 2, 'down': 1},
                                                                                    'response',
                                                                                    'response_int')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {2: 1, 1: 2},
                                                                                    'stimuli_type',
                                                                                    'stimuli_type')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'True': 1, 'False': 0},
                                                                                    'subjectively_correct',
                                                                                    'subjectively_correct_int')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'True': 1, 'False': 0},
                                                                                    'subjectively_correct_one_back',
                                                                                    'subjectively_correct_one_back_int')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'True': 1, 'False': 0},
                                                                                    'objectively_correct',
                                                                                    'objectively_correct_int')

combined_results_dataframe['subject_id'] = combined_results_dataframe['subject_id'].str.replace(r'\D', '', regex=True)
combined_results_dataframe['session'] = combined_results_dataframe['session'].str.replace(r'\D', '', regex=True)

# Now export the data

In [13]:
combined_results_dataframe.to_csv(f'{output_dir}/combined_dataframe_for_r.csv', sep=';', index_label='index')